# 07 - Build Product Search Indexes

- Canonicalize products to one row per product ID.
- Build FAISS and Tantivy indexes over product metadata.

In [1]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.config import load_project_config
from src.rag.preprocessing.processor import TextProcessor
from src.rag.embedding.factory import EmbeddingFactory
from src.rag.product_search import (
    build_canonical_products_file,
    ProductFAISSIndex,
    ProductBM25Index,
)


In [2]:
PRODUCTS_PATH = PROJECT_ROOT / "data" / "processed" / "products_clean.parquet"
CANONICAL_PATH = PROJECT_ROOT / "data" / "processed" / "products_search.parquet"

DENSE_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "products_embedding"
SPARSE_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "products_bm25_tantivy"

CONFIG = load_project_config(PROJECT_ROOT)
RAG_CONFIG = CONFIG
SEARCH_CONFIG = CONFIG["product_search"]

processor = TextProcessor()


## Canonical products

- Create the product-level search table.

In [3]:
canonical = build_canonical_products_file(
    input_path=PRODUCTS_PATH,
    output_path=CANONICAL_PATH,
    overwrite=True,
)

print("Canonical products:", len(canonical))
print("Unique IDs:", canonical["id"].nunique())
display(canonical.head())


Canonical products: 948352
Unique IDs: 948352


,id,title_fa,Rate,Rate_cnt,Category1,Category2,Brand,Price,Seller,Is_Fake,min_price_last_month,sub_category,source_row_count,seller_count,search_text
0,7602,دسته بازی دوال شاک مخصوص پلی استیشن 3,74,274,لوازم جانبی کنسول بازی,Unknown,سونی,3290000.0,رسپان مارکت,False,NaN,toys and kids,1,1,دسته بازی دوال شاک مخصوص پلی استیشن 3 دسته باز...
1,12298,ساعت مچی عقربه ای مردانه کاسیو جی شاک GA-110-1ADR,64,21,اکسسوری مردانه,ساعت مردانه,کاسیو,57312000.0,دیجی‌کالا,False,57133000.0,clothe,2,1,ساعت مچی عقربه ای مردانه کاسیو جی شاک GA-110-1...
2,12302,ساعت مچی دیجیتالی مردانه کاسیو جی شاک GD-100-1BDR,90,8,اکسسوری مردانه,ساعت مردانه,کاسیو,44154000.0,دیجی‌کالا,False,43063000.0,clothe,1,1,ساعت مچی دیجیتالی مردانه کاسیو جی شاک GD-100-1...
3,12423,آلبوم موسیقی همین - رضا صادقی,82,27,موسیقی با کلام,Unknown,آوای هنر,270000.0,نوین رایانه جانبی,False,NaN,book & stationary & art,1,1,آلبوم موسیقی همین - رضا صادقی آلبوم موسیقی همی...
4,15327,تب سنج دیجیتال بیورر مدل FT90,82,92,تب سنج و دماسنج,Unknown,بیورر,17250000.0,پزشکی اکسیژن,False,17100000.0,beauty,1,1,تب سنج دیجیتال بیورر مدل FT90 تب سنج دیجیتال ب...


## Dense product index

- Build the FAISS metadata index.

In [4]:
embedding_model = EmbeddingFactory.create(
    provider=RAG_CONFIG["embedding"]["provider"],
    model_name=RAG_CONFIG["embedding"]["model"],
)

index_cfg = SEARCH_CONFIG["indexing"]

start = time.perf_counter()

dense_manifest = ProductFAISSIndex.build_from_parquet(
    input_path=CANONICAL_PATH,
    output_path=DENSE_INDEX_PATH,
    embedding_model=embedding_model,
    processor=processor,
    chunk_size=index_cfg["dense_chunk_size"],
    encode_batch_size=index_cfg["dense_encode_batch_size"],
    overwrite=True,
)

print("Dense build minutes:", round((time.perf_counter()-start)/60, 2))
print(dense_manifest)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Indexed 10,000 products
Indexed 20,000 products
Indexed 30,000 products
Indexed 40,000 products
Indexed 50,000 products
Indexed 60,000 products
Indexed 70,000 products
Indexed 80,000 products
Indexed 90,000 products
Indexed 100,000 products
Indexed 110,000 products
Indexed 120,000 products
Indexed 130,000 products
Indexed 140,000 products
Indexed 150,000 products
Indexed 160,000 products
Indexed 170,000 products
Indexed 180,000 products
Indexed 190,000 products
Indexed 200,000 products
Indexed 210,000 products
Indexed 220,000 products
Indexed 230,000 products
Indexed 240,000 products
Indexed 250,000 products
Indexed 260,000 products
Indexed 270,000 products
Indexed 280,000 products
Indexed 290,000 products
Indexed 300,000 products
Indexed 310,000 products
Indexed 320,000 products
Indexed 330,000 products
Indexed 340,000 products
Indexed 350,000 products
Indexed 360,000 products
Indexed 370,000 products
Indexed 380,000 products
Indexed 390,000 products
Indexed 400,000 products
Indexed 4

## Sparse product index

- Build the Tantivy metadata index.

In [5]:
start = time.perf_counter()

sparse_manifest = ProductBM25Index.build_from_parquet(
    input_path=CANONICAL_PATH,
    output_path=SPARSE_INDEX_PATH,
    processor=processor,
    batch_size=index_cfg["sparse_batch_size"],
    writer_heap_size=index_cfg["sparse_writer_heap_size"],
    num_threads=index_cfg["sparse_num_threads"],
    overwrite=True,
)

print("Sparse build minutes:", round((time.perf_counter()-start)/60, 2))
print(sparse_manifest)


Indexed 50,000 products
Indexed 100,000 products
Indexed 150,000 products
Indexed 200,000 products
Indexed 250,000 products
Indexed 300,000 products
Indexed 350,000 products
Indexed 400,000 products
Indexed 450,000 products
Indexed 500,000 products
Indexed 550,000 products
Indexed 600,000 products
Indexed 650,000 products
Indexed 700,000 products
Indexed 750,000 products
Indexed 800,000 products
Indexed 850,000 products
Indexed 900,000 products
Indexed 948,352 products
Sparse build minutes: 0.14
{'backend': 'tantivy', 'num_documents': 948352}
